In [ ]:
import pandas as pd
import numpy as np
import os
SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
naps_csv = r"/Users/foramkamdar/Desktop/oehrn_lab/repos/pd_nonmotor/stimuli/all_NAPs_ratings.csv"
df_naps = pd.read_csv(naps_csv)
df = df_naps.copy()
print(df.head())

out_path = r"/Users/foramkamdar/Desktop/oehrn_lab/repos/pd_nonmotor/stimuli/blocks/pdnm002_sessions/future sessions"

In [ ]:
# Define parameters
total_sessions = 6
n_trials_per_session = 112
instruction_cue = ["FEEL", "TONE"]

# Define valence bins and labels
bins = [1, 2, 3, 4, 5, 6, 7, 8, 9]
bin_labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins) - 1)]
n_bins = len(bin_labels) # 8

# Check now many images fall into each valence bin
df["val_bin"] = pd.cut(
    df["Valence"],
    bins=bins,
    labels=bin_labels,
    include_lowest=True
)

counts = df["val_bin"].value_counts().reindex(bin_labels)
print(counts)

# Images available for each session for each valence bin
count_per_session = counts // total_sessions
print(count_per_session)

In [ ]:
# category of images
cats = df["Category"].unique().tolist()
print("Categories of images:", cats)
n_cats = len(cats)
print("Number of categories:", n_cats) 

In [ ]:
# Minimum number of images needed per session for each valence bin
min_trials_per_bin = 14 
min_trials_per_bin_per_session = min_trials_per_bin * total_sessions

# find bins that does not have enough images for each session
bins_insufficient = count_per_session[count_per_session < min_trials_per_bin].index.tolist()
print("Bins with insufficient images for each session:", bins_insufficient)

#set aside insufficient bins for now
df_sufficient = df[~df["val_bin"].isin(bins_insufficient)]
print("Number of images in sufficient bins:", len(df_sufficient))

bins_sufficient = df_sufficient["val_bin"].unique().tolist()
print("Sufficient bins:", bins_sufficient)


In [ ]:
# import os

# rng = np.random.default_rng(SEED)

# df_work = df.copy()

# df_suff = df_work[df_work["val_bin"].isin(bins_sufficient)].copy()
# df_insuff = df_work[df_work["val_bin"].isin(bins_insufficient)].copy()

# # Shuffle once
# df_suff = df_suff.sample(frac=1, random_state=SEED).reset_index(drop=True)
# df_insuff = df_insuff.sample(frac=1, random_state=SEED).reset_index(drop=True)

# used_ids = set()
# all_blocks = []

# base_per_bin = 14

# def sample_balanced(pool, n_take):
#     """Round-robin category sampling"""
#     result = []
#     cats = pool["Category"].unique().tolist()
#     cat_pools = {c: pool[pool["Category"] == c].copy() for c in cats}

#     # pointers
#     for c in cats:
#         cat_pools[c] = cat_pools[c].reset_index(drop=True)

#     i = 0
#     while len(result) < n_take:
#         c = cats[i % len(cats)]
#         if len(cat_pools[c]) > 0:
#             row = cat_pools[c].iloc[0]
#             result.append(row)
#             cat_pools[c] = cat_pools[c].iloc[1:]
#         i += 1

#     return pd.DataFrame(result)

# for s in range(total_sessions):

#     print(f"\n========== SESSION {s+1} ==========")

#     session_rows = []
#     extra_needed = 0

#     # -------------------------
#     # 1. INSUFFICIENT BINS
#     # -------------------------
#     for b in bins_insufficient:
#         pool = df_insuff[(df_insuff["val_bin"] == b) & (~df_insuff["ID"].isin(used_ids))]

#         take_n = int(np.ceil(len(df_insuff[df_insuff["val_bin"] == b]) / total_sessions))
#         take = pool.head(take_n)

#         print(f"\n[INSUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#         deficit = base_per_bin - len(take)
#         if deficit > 0:
#             extra_needed += deficit

#     print(f"\nTotal deficit to redistribute: {extra_needed}")

#     # -------------------------
#     # 2. SUFFICIENT BINS
#     # -------------------------
#     bins_extra_distribution = {b: base_per_bin for b in bins_sufficient}

#     if extra_needed > 0:
#         per_bin_extra = extra_needed // len(bins_sufficient)
#         remainder = extra_needed % len(bins_sufficient)

#         for b in bins_sufficient:
#             bins_extra_distribution[b] += per_bin_extra

#         for b in bins_sufficient[:remainder]:
#             bins_extra_distribution[b] += 1

#     print("\nBin allocation (after redistribution):")
#     print(bins_extra_distribution)

#     # sampling with category balancing
#     for b in bins_sufficient:
#         pool = df_suff[(df_suff["val_bin"] == b) & (~df_suff["ID"].isin(used_ids))]

#         take_n = bins_extra_distribution[b]

#         take = sample_balanced(pool, take_n)

#         print(f"\n[SUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#     # -------------------------
#     # 3. COMBINE
#     # -------------------------
#     block = pd.concat(session_rows).reset_index(drop=True)

#     print("\n--- Combined BEFORE condition split ---")
#     print(block.groupby("val_bin").size())

#     # -------------------------
#     # 4. CONDITION SPLIT
#     # -------------------------
#     block["condition"] = None

#     for b in block["val_bin"].unique():
#         idx = block[block["val_bin"] == b].index.tolist()
#         rng.shuffle(idx)

#         half = len(idx) // 2
#         feel_idx = idx[:half]
#         tone_idx = idx[half:]

#         block.loc[feel_idx, "condition"] = "FEEL"
#         block.loc[tone_idx, "condition"] = "TONE"

#     print("\n--- After condition split ---")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 5. RANDOMIZE
#     # -------------------------
#     block = block.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

#     # -------------------------
#     # 6. FINAL FORMAT
#     # -------------------------
#     block["trial"] = np.arange(1, len(block) + 1)
#     block["filename"] = block["ID"] + ".jpg"

#     block = block[["trial","ID","filename","Category","Valence","val_bin","condition"]]

#     # -------------------------
#     # 7. FINAL SUMMARY
#     # -------------------------
#     print("\n=== FINAL SUMMARY ===")
#     print("Total trials:", len(block))
#     print("\nPer bin:")
#     print(block.groupby("val_bin").size())
#     print("\nPer bin x condition:")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 8. SAVE
#     # -------------------------
#     out_csv = os.path.join(out_path, f"session_{s+1}.csv")
#     block.to_csv(out_csv, index=False)

#     print("\nSaved:", out_csv)

#     all_blocks.append(block)

In [ ]:
# import os

# rng = np.random.default_rng(SEED)

# df_work = df.copy()

# df_suff = df_work[df_work["val_bin"].isin(bins_sufficient)].copy()
# df_insuff = df_work[df_work["val_bin"].isin(bins_insufficient)].copy()

# # Shuffle once
# df_suff = df_suff.sample(frac=1, random_state=SEED).reset_index(drop=True)
# df_insuff = df_insuff.sample(frac=1, random_state=SEED).reset_index(drop=True)

# used_ids = set()
# all_blocks = []

# base_per_bin = 14

# # global tracker
# category_usage = {c: 0 for c in cats}

# def sample_balanced(pool, n_take):
#     result = []

#     pool = pool.copy()

#     while len(result) < n_take:
#         # sort categories by least used globally
#         sorted_cats = sorted(category_usage.keys(), key=lambda x: category_usage[x])

#         picked = False

#         for c in sorted_cats:
#             sub = pool[pool["Category"] == c]
#             if len(sub) > 0:
#                 row = sub.iloc[0]
#                 result.append(row)

#                 # update
#                 category_usage[c] += 1
#                 pool = pool.drop(row.name)

#                 picked = True
#                 break

#         if not picked:
#             raise ValueError("Not enough images for category balancing")

#     return pd.DataFrame(result)

# # Precompute insufficient bin distribution
# insuff_distribution = {}

# for b in bins_insufficient:
#     total = len(df_insuff[df_insuff["val_bin"] == b])
#     base = total // total_sessions
#     remainder = total % total_sessions

#     dist = [base] * total_sessions
#     for i in range(remainder):
#         dist[i] += 1

#     rng.shuffle(dist)  # avoid fixed bias
#     insuff_distribution[b] = dist

# print("\nInsufficient bin distribution:", insuff_distribution)

# for s in range(total_sessions):

#     print(f"\n========== SESSION {s+1} ==========")

#     session_rows = []
#     extra_needed = 0

#     # -------------------------
#     # 1. INSUFFICIENT BINS
#     # -------------------------
#     for b in bins_insufficient:
#         pool = df_insuff[(df_insuff["val_bin"] == b) & (~df_insuff["ID"].isin(used_ids))]

#         take_n = insuff_distribution[b][s]
#         take = sample_balanced(pool, b, take_n)

#         print(f"\n[INSUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#         deficit = base_per_bin - len(take)
#         if deficit > 0:
#             extra_needed += deficit

#     print(f"\nTotal deficit to redistribute: {extra_needed}")

#     # -------------------------
#     # 2. SUFFICIENT BINS
#     # -------------------------
#     bins_extra_distribution = {b: base_per_bin for b in bins_sufficient}

#     if extra_needed > 0:
#         per_bin_extra = extra_needed // len(bins_sufficient)
#         remainder = extra_needed % len(bins_sufficient)

#     for b in bins_sufficient:
#         bins_extra_distribution[b] += per_bin_extra

#     # ROTATE instead of fixed order
#     rotated_bins = bins_sufficient[s % len(bins_sufficient):] + bins_sufficient[:s % len(bins_sufficient)]

#     for b in rotated_bins[:remainder]:
#         bins_extra_distribution[b] += 1

#     print("\nBin allocation (after redistribution):")
#     print(bins_extra_distribution)

#     # sampling with category balancing
#     for b in bins_sufficient:
#         pool = df_suff[(df_suff["val_bin"] == b) & (~df_suff["ID"].isin(used_ids))]

#         take_n = bins_extra_distribution[b]

#         take = sample_balanced(pool, take_n)

#         print(f"\n[SUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#     # -------------------------
#     # 3. COMBINE
#     # -------------------------
#     block = pd.concat(session_rows).reset_index(drop=True)

#     print("\n--- Combined BEFORE condition split ---")
#     print(block.groupby("val_bin").size())

#     # -------------------------
#     # 4. CONDITION SPLIT
#     # -------------------------
#     block["condition"] = None

#     for b in block["val_bin"].unique():
#         idx = block[block["val_bin"] == b].index.tolist()
#         rng.shuffle(idx)

#         half = len(idx) // 2
#         feel_idx = idx[:half]
#         tone_idx = idx[half:]

#         block.loc[feel_idx, "condition"] = "FEEL"
#         block.loc[tone_idx, "condition"] = "TONE"

#     print("\n--- After condition split ---")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 5. RANDOMIZE
#     # -------------------------
#     block = block.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

#     # -------------------------
#     # 6. FINAL FORMAT
#     # -------------------------
#     block["trial"] = np.arange(1, len(block) + 1)
#     block["filename"] = block["ID"] + ".jpg"

#     block = block[["trial","ID","filename","Category","Valence","val_bin","condition"]]

#     # -------------------------
#     # 7. FINAL SUMMARY
#     # -------------------------
#     print("\n=== FINAL SUMMARY ===")
#     print("Total trials:", len(block))
#     print("\nPer bin:")
#     print(block.groupby("val_bin").size())
#     print("\nPer bin x condition:")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 8. SAVE
#     # -------------------------
#     out_csv = os.path.join(out_path, f"session_{s+1}.csv")
#     block.to_csv(out_csv, index=False)

#     print("\nSaved:", out_csv)

#     all_blocks.append(block)

In [ ]:
# #BINS BALANCED: SAFE TO USE. ADDING CATEOGPRY LATER
# SEED = 42
# rng = np.random.default_rng(SEED)

# df_work = df.copy()

# df_suff = df_work[df_work["val_bin"].isin(bins_sufficient)].copy()
# df_insuff = df_work[df_work["val_bin"].isin(bins_insufficient)].copy()

# # Shuffle once
# df_suff = df_suff.sample(frac=1, random_state=SEED).reset_index(drop=True)
# df_insuff = df_insuff.sample(frac=1, random_state=SEED).reset_index(drop=True)

# used_ids = set()
# all_blocks = []

# base_per_bin = 14

# # =========================================================
# # 🔴 FIX 1: PRECOMPUTE INSUFFICIENT BIN DISTRIBUTION
# # =========================================================
# insuff_distribution = {}

# for b in bins_insufficient:
#     total = len(df_insuff[df_insuff["val_bin"] == b])
#     base = total // total_sessions
#     remainder = total % total_sessions

#     dist = [base] * total_sessions
#     for i in range(remainder):
#         dist[i] += 1

#     rng.shuffle(dist)
#     insuff_distribution[b] = dist

# print("\nInsufficient bin distribution:", insuff_distribution)

# # =========================================================
# # 🔴 FIX 2: PER-BIN CATEGORY TRACKING (CRITICAL FIX)
# # =========================================================
# bin_cat_used = {
#     b: {c: 0 for c in cats} for b in bin_labels
# }

# def sample_balanced(pool, b, n_take):
#     result = []
#     pool = pool.copy()

#     while len(result) < n_take:

#         # prioritize least-used category within THIS BIN
#         sorted_cats = sorted(cats, key=lambda c: bin_cat_used[b][c])

#         picked = False

#         for c in sorted_cats:
#             sub = pool[pool["Category"] == c]

#             if len(sub) > 0:
#                 row = sub.iloc[0]

#                 result.append(row)

#                 # update tracker
#                 bin_cat_used[b][c] += 1
#                 pool = pool.drop(row.name)

#                 picked = True
#                 break

#         if not picked:
#             raise ValueError(f"Not enough images for bin {b}")

#     return pd.DataFrame(result)

# # =========================================================
# # MAIN LOOP
# # =========================================================
# for s in range(total_sessions):

#     print(f"\n========== SESSION {s+1} ==========")

#     session_rows = []
#     extra_needed = 0

#     # -------------------------
#     # 1. INSUFFICIENT BINS
#     # -------------------------
#     for b in bins_insufficient:
#         pool = df_insuff[
#             (df_insuff["val_bin"] == b) &
#             (~df_insuff["ID"].isin(used_ids))
#         ]

#         take_n = insuff_distribution[b][s]

#         take = sample_balanced(pool, b, take_n)

#         print(f"\n[INSUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#         deficit = base_per_bin - len(take)
#         if deficit > 0:
#             extra_needed += deficit

#     print(f"\nTotal deficit to redistribute: {extra_needed}")

#     # -------------------------
#     # 2. SUFFICIENT BINS
#     # -------------------------
#     bins_extra_distribution = {b: base_per_bin for b in bins_sufficient}

#     if extra_needed > 0:
#         per_bin_extra = extra_needed // len(bins_sufficient)
#         remainder = extra_needed % len(bins_sufficient)

#         for b in bins_sufficient:
#             bins_extra_distribution[b] += per_bin_extra

#         # 🔴 FIX: rotate to remove bias
#         rotated_bins = (
#             bins_sufficient[s % len(bins_sufficient):] +
#             bins_sufficient[:s % len(bins_sufficient)]
#         )

#         for b in rotated_bins[:remainder]:
#             bins_extra_distribution[b] += 1

#     print("\nBin allocation (after redistribution):")
#     print(bins_extra_distribution)

#     # -------------------------
#     # 3. SUFFICIENT BIN SAMPLING
#     # -------------------------
#     for b in bins_sufficient:
#         pool = df_suff[
#             (df_suff["val_bin"] == b) &
#             (~df_suff["ID"].isin(used_ids))
#         ]

#         take_n = bins_extra_distribution[b]

#         take = sample_balanced(pool, b, take_n)

#         print(f"\n[SUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#     # -------------------------
#     # 4. COMBINE
#     # -------------------------
#     block = pd.concat(session_rows).reset_index(drop=True)

#     print("\n--- Combined BEFORE condition split ---")
#     print(block.groupby("val_bin").size())

#     # -------------------------
#     # 5. CONDITION SPLIT
#     # -------------------------
#     block["condition"] = None

#     for b in block["val_bin"].unique():
#         idx = block[block["val_bin"] == b].index.tolist()
#         rng.shuffle(idx)

#         half = len(idx) // 2
#         block.loc[idx[:half], "condition"] = "FEEL"
#         block.loc[idx[half:], "condition"] = "TONE"

#     print("\n--- After condition split ---")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 6. RANDOMIZE
#     # -------------------------
#     block = block.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

#     # -------------------------
#     # 7. FINAL FORMAT
#     # -------------------------
#     block["trial"] = np.arange(1, len(block) + 1)
#     block["filename"] = block["ID"] + ".jpg"

#     block = block[
#         ["trial","ID","filename","Category","Valence","val_bin","condition"]
#     ]

#     # -------------------------
#     # 8. FINAL SUMMARY
#     # -------------------------
#     print("\n=== FINAL SUMMARY ===")
#     print("Total trials:", len(block))
#     print("\nPer bin:")
#     print(block.groupby("val_bin").size())
#     print("\nPer bin x condition:")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 9. SAVE
#     # -------------------------
#     out_csv = os.path.join(out_path, f"session_{s+1}.csv")
#     block.to_csv(out_csv, index=False)

#     print("\nSaved:", out_csv)

#     all_blocks.append(block)

# # =========================================================
# # 🔬 FINAL GLOBAL CHECK (IMPORTANT)
# # =========================================================
# all_data = pd.concat(all_blocks)

# print("\n================ GLOBAL CHECK ================")
# print("\nPer bin:")
# print(all_data.groupby("val_bin").size())

# print("\nPer category:")
# print(all_data.groupby("Category").size())

# print("\nPer bin x category:")
# print(all_data.groupby(["val_bin","Category"]).size())

In [ ]:
# #BINS BALANCED: SAFE TO USE. no category balancing. Totally random within bins. will be biased like dataset.
# SEED = 42
# rng = np.random.default_rng(SEED)

# df_work = df.copy()

# df_suff = df_work[df_work["val_bin"].isin(bins_sufficient)].copy()
# df_insuff = df_work[df_work["val_bin"].isin(bins_insufficient)].copy()

# # Shuffle once
# df_suff = df_suff.sample(frac=1, random_state=SEED).reset_index(drop=True)
# df_insuff = df_insuff.sample(frac=1, random_state=SEED).reset_index(drop=True)

# used_ids = set()
# all_blocks = []

# base_per_bin = 14

# # =========================================================
# #  PRECOMPUTE INSUFFICIENT BIN DISTRIBUTION
# # =========================================================
# insuff_distribution = {}

# for b in bins_insufficient:
#     total = len(df_insuff[df_insuff["val_bin"] == b])
#     base = total // total_sessions
#     remainder = total % total_sessions

#     dist = [base] * total_sessions
#     for i in range(remainder):
#         dist[i] += 1

#     rng.shuffle(dist)
#     insuff_distribution[b] = dist

# print("\nInsufficient bin distribution:", insuff_distribution)



# # =========================================================
# # MAIN LOOP
# # =========================================================
# for s in range(total_sessions):

#     print(f"\n========== SESSION {s+1} ==========")

#     session_rows = []
#     extra_needed = 0

#     # -------------------------
#     # 1. INSUFFICIENT BINS
#     # -------------------------
#     for b in bins_insufficient:
#         pool = df_insuff[
#             (df_insuff["val_bin"] == b) &
#             (~df_insuff["ID"].isin(used_ids))
#         ]

#         take_n = insuff_distribution[b][s]

#         #take = sample_balanced(pool, b, take_n)
#         take = pool.sample(n=take_n, random_state=SEED + s)

#         print(f"\n[INSUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#         deficit = base_per_bin - len(take)
#         if deficit > 0:
#             extra_needed += deficit

#     print(f"\nTotal deficit to redistribute: {extra_needed}")

#     # -------------------------
#     # 2. SUFFICIENT BINS
#     # -------------------------
#     bins_extra_distribution = {b: base_per_bin for b in bins_sufficient}

#     if extra_needed > 0:
#         per_bin_extra = extra_needed // len(bins_sufficient)
#         remainder = extra_needed % len(bins_sufficient)

#         for b in bins_sufficient:
#             bins_extra_distribution[b] += per_bin_extra

#         # 🔴 FIX: rotate to remove bias
#         rotated_bins = (
#             bins_sufficient[s % len(bins_sufficient):] +
#             bins_sufficient[:s % len(bins_sufficient)]
#         )

#         for b in rotated_bins[:remainder]:
#             bins_extra_distribution[b] += 1

#     print("\nBin allocation (after redistribution):")
#     print(bins_extra_distribution)

#     # -------------------------
#     # 3. SUFFICIENT BIN SAMPLING
#     # -------------------------
#     for b in bins_sufficient:
#         pool = df_suff[
#             (df_suff["val_bin"] == b) &
#             (~df_suff["ID"].isin(used_ids))
#         ]

#         take_n = bins_extra_distribution[b]

#         take = pool.sample(n=take_n, random_state=SEED + s)

#         print(f"\n[SUFF BIN {b}] taking {len(take)}")
#         print(take[["ID","Category","val_bin"]])

#         used_ids.update(take["ID"].tolist())
#         session_rows.append(take)

#     # -------------------------
#     # 4. COMBINE
#     # -------------------------
#     block = pd.concat(session_rows).reset_index(drop=True)

#     print("\n--- Combined BEFORE condition split ---")
#     print(block.groupby("val_bin").size())

#     # -------------------------
#     # 5. CONDITION SPLIT
#     # -------------------------
#     block["condition"] = None

#     for b in block["val_bin"].unique():
#         idx = block[block["val_bin"] == b].index.tolist()
#         rng.shuffle(idx)

#         half = len(idx) // 2
#         block.loc[idx[:half], "condition"] = "FEEL"
#         block.loc[idx[half:], "condition"] = "TONE"

#     print("\n--- After condition split ---")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 6. RANDOMIZE
#     # -------------------------
#     block = block.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

#     # -------------------------
#     # 7. FINAL FORMAT
#     # -------------------------
#     block["trial"] = np.arange(1, len(block) + 1)
#     block["filename"] = block["ID"] + ".jpg"

#     block = block[
#         ["trial","ID","filename","Category","Valence","val_bin","condition"]
#     ]

#     # -------------------------
#     # 8. FINAL SUMMARY
#     # -------------------------
#     print("\n=== FINAL SUMMARY ===")
#     print("Total trials:", len(block))
#     print("\nPer bin:")
#     print(block.groupby("val_bin").size())
#     print("\nPer bin x condition:")
#     print(block.groupby(["val_bin","condition"]).size())

#     # -------------------------
#     # 9. SAVE
#     # -------------------------
#     out_csv = os.path.join(out_path, f"session_{s+1}.csv")
#     block.to_csv(out_csv, index=False)

#     print("\nSaved:", out_csv)

#     all_blocks.append(block)

# # =========================================================
# # 🔬 FINAL GLOBAL CHECK (IMPORTANT)
# # =========================================================
# all_data = pd.concat(all_blocks)

# print("\n================ GLOBAL CHECK ================")
# print("\nPer bin:")
# print(all_data.groupby("val_bin").size())

# print("\nPer category:")
# print(all_data.groupby("Category").size())

# print("\nPer bin x category:")
# print(all_data.groupby(["val_bin","Category"]).size())

In [ ]:
#Soft category balancing.


df_work = df.copy()

df_suff = df_work[df_work["val_bin"].isin(bins_sufficient)].copy()
df_insuff = df_work[df_work["val_bin"].isin(bins_insufficient)].copy()

# Shuffle once
df_suff = df_suff.sample(frac=1, random_state=SEED).reset_index(drop=True)
df_insuff = df_insuff.sample(frac=1, random_state=SEED).reset_index(drop=True)

used_ids = set()
all_blocks = []

base_per_bin = 14

# =========================================================
#  PRECOMPUTE INSUFFICIENT BIN DISTRIBUTION
# =========================================================
insuff_distribution = {}

for b in bins_insufficient:
    total = len(df_insuff[df_insuff["val_bin"] == b])
    base = total // total_sessions
    remainder = total % total_sessions

    dist = [base] * total_sessions
    for i in range(remainder):
        dist[i] += 1

    rng.shuffle(dist)
    insuff_distribution[b] = dist

print("\nInsufficient bin distribution:", insuff_distribution)


def soft_category_sample(pool, n_take, seed):
    rng_local = np.random.default_rng(seed)

    # shuffle pool first
    pool = pool.sample(frac=1, random_state=seed).copy()

    # group by category
    cat_groups = {
        c: pool[pool["Category"] == c].copy()
        for c in pool["Category"].unique()
    }

    # shuffle each category
    for c in cat_groups:
        cat_groups[c] = cat_groups[c].sample(frac=1, random_state=seed)

    result = []

    # round-robin pick (SOFT, not forced)
    while len(result) < n_take:
        progressed = False

        for c in list(cat_groups.keys()):
            if len(cat_groups[c]) > 0:
                row = cat_groups[c].iloc[0]
                result.append(row)

                cat_groups[c] = cat_groups[c].iloc[1:]
                progressed = True

                if len(result) == n_take:
                    break

        if not progressed:
            break  # no more data

    return pd.DataFrame(result)



# =========================================================
# MAIN LOOP
# =========================================================
for s in range(total_sessions):

    print(f"\n========== SESSION {s+1} ==========")

    session_rows = []
    extra_needed = 0

    # -------------------------
    # 1. INSUFFICIENT BINS
    # -------------------------
    for b in bins_insufficient:
        pool = df_insuff[
            (df_insuff["val_bin"] == b) &
            (~df_insuff["ID"].isin(used_ids))
        ]

        take_n = insuff_distribution[b][s]

        take = soft_category_sample(pool, take_n, SEED + s)

        print(f"\n[INSUFF BIN {b}] taking {len(take)}")
        print(take[["ID","Category","val_bin"]])

        used_ids.update(take["ID"].tolist())
        session_rows.append(take)

        deficit = base_per_bin - len(take)
        if deficit > 0:
            extra_needed += deficit

    print(f"\nTotal deficit to redistribute: {extra_needed}")

    # -------------------------
    # 2. SUFFICIENT BINS
    # -------------------------
    bins_extra_distribution = {b: base_per_bin for b in bins_sufficient}

    if extra_needed > 0:
        per_bin_extra = extra_needed // len(bins_sufficient)
        remainder = extra_needed % len(bins_sufficient)

        for b in bins_sufficient:
            bins_extra_distribution[b] += per_bin_extra

        # 🔴 FIX: rotate to remove bias
        rotated_bins = (
            bins_sufficient[s % len(bins_sufficient):] +
            bins_sufficient[:s % len(bins_sufficient)]
        )

        for b in rotated_bins[:remainder]:
            bins_extra_distribution[b] += 1

    print("\nBin allocation (after redistribution):")
    print(bins_extra_distribution)

    # -------------------------
    # 3. SUFFICIENT BIN SAMPLING
    # -------------------------
    for b in bins_sufficient:
        pool = df_suff[
            (df_suff["val_bin"] == b) &
            (~df_suff["ID"].isin(used_ids))
        ]

        take_n = bins_extra_distribution[b]

        take = soft_category_sample(pool, take_n, SEED + s)

        print(f"\n[SUFF BIN {b}] taking {len(take)}")
        print(take[["ID","Category","val_bin"]])

        used_ids.update(take["ID"].tolist())
        session_rows.append(take)

    # -------------------------
    # 4. COMBINE
    # -------------------------
    block = pd.concat(session_rows).reset_index(drop=True)

    print("\n--- Combined BEFORE condition split ---")
    print(block.groupby("val_bin").size())

    # -------------------------
    # 5. CONDITION SPLIT
    # -------------------------
    block["condition"] = None

    for b in block["val_bin"].unique():
        idx = block[block["val_bin"] == b].index.tolist()
        rng.shuffle(idx)

        half = len(idx) // 2
        block.loc[idx[:half], "condition"] = "FEEL"
        block.loc[idx[half:], "condition"] = "TONE"

    print("\n--- After condition split ---")
    print(block.groupby(["val_bin","condition"]).size())

    # -------------------------
    # 6. RANDOMIZE
    # -------------------------
    block = block.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

    # -------------------------
    # 7. FINAL FORMAT
    # -------------------------
    block["trial"] = np.arange(1, len(block) + 1)
    block["filename"] = block["ID"] + ".jpg"

    block = block[
        ["trial","ID","filename","Category","Valence","val_bin","condition"]
    ]

    # -------------------------
    # 8. FINAL SUMMARY
    # -------------------------
    print("\n=== FINAL SUMMARY ===")
    print("Total trials:", len(block))
    print("\nPer bin:")
    print(block.groupby("val_bin").size())
    print("\nPer bin x condition:")
    print(block.groupby(["val_bin","condition"]).size())

    # -------------------------
    # 9. SAVE
    # -------------------------
    out_csv = os.path.join(out_path, f"session_{s+1}.csv")
    block.to_csv(out_csv, index=False)

    print("\nSaved:", out_csv)

    all_blocks.append(block)

# =========================================================
# 🔬 FINAL GLOBAL CHECK (IMPORTANT)
# =========================================================
all_data = pd.concat(all_blocks)

print("\n================ GLOBAL CHECK ================")
print("\nPer bin:")
print(all_data.groupby("val_bin").size())

print("\nPer category:")
print(all_data.groupby("Category").size())

print("\nPer bin x category:")
print(all_data.groupby(["val_bin","Category"]).size())

In [ ]:
print("\nPer session x category:")
for i, block in enumerate(all_blocks):
    print(f"\nSession {i+1}")
    print(block.groupby("Category").size())

In [ ]:
print("\nPer session x bins:")
for i, block in enumerate(all_blocks):
    print(f"\nSession {i+1}")

    # use fixed bin order
    for b in bin_labels:
        bin_df = block[block["val_bin"] == b]
        if bin_df.empty:
            continue

        total = len(bin_df)
        split = (
            bin_df["condition"]
            .value_counts()
            .reindex(["FEEL", "TONE"], fill_value=0)
        )

        print(f"Bin {b}: {total}, split")
        #print(split)
